<a href="https://colab.research.google.com/github/2410010585amel/STATIS/blob/main/P02_STATIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, permutations, combinations
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print("Library berhasil di-import!")

Library berhasil di-import!


In [2]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

S_indices = df.index.values
total_samples = len(df)

print(f"Total Ruang Sampel |S|: {total_samples} sampel")
print("Distribusi Kelas di Ruang Sampel S:")
print(df['species'].value_counts())
df.head()

Total Ruang Sampel |S|: 150 sampel
Distribusi Kelas di Ruang Sampel S:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [3]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['target'])

train_idx = train_df.index.values
test_idx = test_df.index.values

union_idx = np.union1d(train_idx, test_idx)
intersect_idx = np.intersect1d(train_idx, test_idx)

print(f"Jumlah Data Train (|Train|): {len(train_idx)}")
print(f"Jumlah Data Test (|Test|): {len(test_idx)}")
print(f"Apakah Train U Test == S? {len(union_idx) == total_samples}")
print(f"Apakah Train n Test == Kosong? {len(intersect_idx) == 0}")

Jumlah Data Train (|Train|): 105
Jumlah Data Test (|Test|): 45
Apakah Train U Test == S? True
Apakah Train n Test == Kosong? True


In [4]:
E1 = df[df['sepal length (cm)'] > 6.0]
p_E1 = len(E1) / total_samples

E2 = df[df['sepal length (cm)'] <= 6.0]
p_E2 = len(E2) / total_samples

assert p_E1 >= 0 and p_E2 >= 0, "Pelanggaran Aksioma 1!"

p_S = len(df) / total_samples

p_E1_union_E2 = len(df[(df['sepal length (cm)'] > 6.0) | (df['sepal length (cm)'] <= 6.0)]) / total_samples

print(f"--- PEMBUKTIAN AKSIOMA KOLMOGOROV ---")
print(f"Aksioma 1 (Non-negatif) : P(E1) = {p_E1:.4f} >= 0 [TERBUKTI]")
print(f"Aksioma 2 (Normalisasi) : P(S)  = {p_S:.4f} [TERBUKTI]")
print(f"Aksioma 3 (Aditivitas)  : P(E1 U E2) = {p_E1_union_E2:.4f} == P(E1)+P(E2) ({p_E1 + p_E2:.4f}) [TERBUKTI]")

--- PEMBUKTIAN AKSIOMA KOLMOGOROV ---
Aksioma 1 (Non-negatif) : P(E1) = 0.4067 >= 0 [TERBUKTI]
Aksioma 2 (Normalisasi) : P(S)  = 1.0000 [TERBUKTI]
Aksioma 3 (Aditivitas)  : P(E1 U E2) = 1.0000 == P(E1)+P(E2) (1.0000) [TERBUKTI]


In [5]:
A = df[df['sepal length (cm)'] > 5.5].index.values
B = df[df['petal length (cm)'] > 4.0].index.values

irisan = np.intersect1d(A, B)
gabungan = np.union1d(A, B)
selisih_A_B = np.setdiff1d(A, B)
xor_A_B = np.setxor1d(A, B)

print(f"Ukuran Himpunan A: {len(A)}")
print(f"Ukuran Himpunan B: {len(B)}")
print(f"Ukuran Irisan (A n B): {len(irisan)}")
print(f"Ukuran Gabungan (A U B): {len(gabungan)}")
print(f"Ukuran Selisih (A \\ B): {len(selisih_A_B)}")
print(f"Ukuran Beda Simetris (A XOR B): {len(xor_A_B)}")

Ukuran Himpunan A: 91
Ukuran Himpunan B: 84
Ukuran Irisan (A n B): 81
Ukuran Gabungan (A U B): 94
Ukuran Selisih (A \ B): 10
Ukuran Beda Simetris (A XOR B): 13


In [6]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

kombinasi = list(product(param_grid['n_neighbors'], param_grid['weights'], param_grid['metric']))

print(f"Kandidat n_neighbors ({len(param_grid['n_neighbors'])}): {param_grid['n_neighbors']}")
print(f"Kandidat weights ({len(param_grid['weights'])}): {param_grid['weights']}")
print(f"Kandidat metric ({len(param_grid['metric'])}): {param_grid['metric']}")
print(f"\nTotal Kombinasi Grid Search (|Search Space|): {len(kombinasi)}")
print("Contoh 5 Kombinasi Pertama:")
for c in kombinasi[:5]:
    print(c)

Kandidat n_neighbors (4): [3, 5, 7, 9]
Kandidat weights (2): ['uniform', 'distance']
Kandidat metric (2): ['euclidean', 'manhattan']

Total Kombinasi Grid Search (|Search Space|): 16
Contoh 5 Kombinasi Pertama:
(3, 'uniform', 'euclidean')
(3, 'uniform', 'manhattan')
(3, 'distance', 'euclidean')
(3, 'distance', 'manhattan')
(5, 'uniform', 'euclidean')


In [7]:
X = df[iris.feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn = KNeighborsClassifier()
grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Hiperparameter Terbaik: {grid_search.best_params_}")
print(f"Akurasi Validasi Silang Terbaik: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f"Akurasi Test Set: {accuracy_score(y_test, y_pred):.4f}")

Hiperparameter Terbaik: {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'uniform'}
Akurasi Validasi Silang Terbaik: 0.9750
Akurasi Test Set: 1.0000


[TUGAS MANDIRI] Soal 1 — Eksperimen Split Data

In [8]:
# Soal 1: Menguji pengaruh random_state pada pembagian indeks data
for rs in [10, 42, 100]:
    tr_df, ts_df = train_test_split(df, test_size=0.3, random_state=rs, stratify=df['target'])
    print(f"random_state = {rs}:")
    print(f"  5 Indeks awal Train: {tr_df.index.values[:5]}")
    print(f"  5 Indeks awal Test : {ts_df.index.values[:5]}\n")

random_state = 10:
  5 Indeks awal Train: [ 4 67 17 87 78]
  5 Indeks awal Test : [  9 125  15 117  55]

random_state = 42:
  5 Indeks awal Train: [ 98  68  19 143  99]
  5 Indeks awal Test : [107  63 133  56 127]

random_state = 100:
  5 Indeks awal Train: [71 90  4 17 18]
  5 Indeks awal Test : [ 34  89 121  94 104]



In [9]:
# Soal 2: Probabilitas Bersyarat P(species = 'virginica' | sepal length > 6.0)
kondisi_B = df[df['sepal length (cm)'] > 6.0]
n_B = len(kondisi_B)

kondisi_A_and_B = kondisi_B[kondisi_B['species'] == 'virginica']
n_A_and_B = len(kondisi_A_and_B)

p_conditional = n_A_and_B / n_B

print(f"Jumlah data dengan sepal length > 6.0 (|B|)             : {n_B}")
print(f"Jumlah data 'virginica' dengan sepal length > 6.0 (|A n B|): {n_A_and_B}")
print(f"P(virginica | sepal length > 6.0)                          : {p_conditional:.4f} ({p_conditional * 100:.2f}%)")

Jumlah data dengan sepal length > 6.0 (|B|)             : 61
Jumlah data 'virginica' dengan sepal length > 6.0 (|A n B|): 41
P(virginica | sepal length > 6.0)                          : 0.6721 (67.21%)


In [10]:
# Soal 3: Verifikasi Kombinatorik Grid Search Baru
param_grid_baru = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree']
}

kombinasi_baru = list(product(
    param_grid_baru['n_neighbors'],
    param_grid_baru['weights'],
    param_grid_baru['metric'],
    param_grid_baru['algorithm']
))

print(f"Total Kombinasi Grid Search Baru: {len(kombinasi_baru)} kombinasi")

Total Kombinasi Grid Search Baru: 48 kombinasi
